# Advanced Method: Generative Modern Hopfield Sampler for Stylized Market Facts

This notebook records the final method used in `L6d` to generate synthetic market returns with a modern Hopfield model.

The focus here is method derivation and interpretation, not implementation details. For executable code, see [L6d Lab Solution](CHEME-5820-L6d-Lab-Solution-Spring-2026.ipynb).

> __Learning Objectives:__
>
> By the end of this notebook, you should be able to:
>
> * __Reconstruct the generator pipeline:__ Identify each state variable in the synthetic-day loop and explain how one new return vector is produced. Track how the query state, probability vector, and generated return vector evolve from one day to the next.
> * __Explain why the sampler can match key stylized facts:__ Distinguish which parts of the method influence tail behavior, linear autocorrelation, and volatility clustering. Connect each empirical behavior to a specific mechanism in the sampling workflow.
> * __Tune the model with controlled trade-offs:__ Describe how persistence and noise parameters change smoothness, clustering strength, and realism. Use a parameter-first view to decide when to favor distribution shape fidelity versus strict moment matching.

Let's get started!

___


## Method Construction

We model each trading day as a feature vector over firms, and generate a synthetic path by iterating a stateful Hopfield-driven map.

> __Notation and Objects:__
>
> Let $F$ be the number of firms and $K$ be the number of historical memory days. The memory matrix is $\boldsymbol{M}\in\mathbb{R}^{F\times K}$ with day vectors in columns, the query state is $\boldsymbol{s}_{t}\in\mathbb{R}^{F}$, and the recovered memory probability is $\boldsymbol{p}_{t}\in\Delta^{K-1}$.

For each synthetic day $t$, we apply the following sequence:

1. __Recover memory probabilities from the current query__
$$
\boldsymbol{p}_{t}=\mathrm{RecoverProbabilities}(\boldsymbol{s}_{t};\beta,\epsilon_{p},\epsilon_{s},\mathrm{maxiterations}).
$$

2. __Build a stochastic top-$k$ proposal__
$$
\tilde{p}_{t,j}=p_{t,j}+\sigma_{\mathrm{select}}\,\varepsilon_{t,j},\qquad \varepsilon_{t,j}\sim\mathcal{N}(0,1),
$$
keep the top-$k$ indices $\mathcal{I}_{t}$ after sorting $\tilde{p}_{t,j}$, then define
$$
q^{\mathrm{new}}_{t,j}=\begin{cases}
p_{t,j}, & j\in\mathcal{I}_{t},\\
0, & j\notin\mathcal{I}_{t},
\end{cases}
\qquad
\boldsymbol{q}^{\mathrm{new}}_{t}\leftarrow\frac{\boldsymbol{q}^{\mathrm{new}}_{t}}{\sum_{j=1}^{K}q^{\mathrm{new}}_{t,j}+10^{-12}}.
$$

3. __Apply sticky weights and sharpening__
$$
\boldsymbol{q}_{t}=\lambda_{q}\,\boldsymbol{q}_{t-1}+(1-\lambda_{q})\,\boldsymbol{q}^{\mathrm{new}}_{t},
\qquad
\boldsymbol{q}_{t}\leftarrow\frac{\boldsymbol{q}_{t}}{\sum_{j=1}^{K}q_{t,j}+10^{-12}},
$$
$$
q_{t,j}\leftarrow q_{t,j}^{\gamma_{\mathrm{sharpen}}},
\qquad
\boldsymbol{q}_{t}\leftarrow\frac{\boldsymbol{q}_{t}}{\sum_{j=1}^{K}q_{t,j}+10^{-12}}.
$$

4. __Generate a base vector and apply latent volatility__
$$
\boldsymbol{m}^{\mathrm{raw}}_{t}=\boldsymbol{M}\boldsymbol{q}_{t}.
$$
Let $\boldsymbol{\mu}_{\mathrm{orig}}\in\mathbb{R}^{F}$ be historical per-ticker means and define
$$
\log h_{t}=\rho_{\log h}\,\log h_{t-1}+\sigma_{\log h}\,\xi_{t},\qquad \xi_{t}\sim\mathcal{N}(0,1),
$$
$$
h_{t}=\exp(\log h_{t}),
\qquad
\boldsymbol{m}_{t}=\boldsymbol{\mu}_{\mathrm{orig}}+\sqrt{h_{t}}\left(\boldsymbol{m}^{\mathrm{raw}}_{t}-\boldsymbol{\mu}_{\mathrm{orig}}\right).
$$

5. __Update the next query state__
$$
\boldsymbol{s}_{t+1}=\lambda_{\mathrm{query}}\,\boldsymbol{s}_{t}+(1-\lambda_{\mathrm{query}})\,\boldsymbol{m}_{t}+\sigma_{\mathrm{query}}\,\boldsymbol{\eta}_{t},
\qquad
\boldsymbol{\eta}_{t}\sim\mathcal{N}(\boldsymbol{0},\boldsymbol{I}_{F}).
$$

6. __Optional calibration layer__

After synthesis, mean matching and partial standard-deviation matching can be applied as post-adjustments. These operations correct marginal moments but do not create volatility clustering on their own.


## Stylized-Facts Interpretation

This section links each stylized fact to a specific mechanism in the sampler.

> __Why each behavior appears:__
>
> * __Fat tails:__ The generated day is a nonlinear map of top-$k$ memory mixtures with stochastic selection and volatility modulation. Regime switching and log-volatility shocks produce heavier tails than a single Gaussian linear model.
> * __Low linear autocorrelation in returns:__ When query persistence is low, return levels do not carry strongly from one day to the next. This keeps first-lag autocorrelation near zero even when the process remains non-Gaussian.
> * __Volatility clustering:__ The latent log-volatility state is persistent, so high-volatility days tend to be followed by high-volatility days. This creates positive autocorrelation in return magnitudes and squared returns.

These diagnostics separate mechanism from outcome: return-level dependence and volatility dependence come from different parameter blocks.


## Practical Tuning Guide

Use the following controls to tune realism in a stable way:

* __Volatility clustering strength (`rho_logh`, `sigma_logh`):__ Increase `rho_logh` to lengthen volatility persistence in time. Increase `sigma_logh` to widen high-volatility episodes.
* __Raw-return memory (`lambda_q`, `lambda_query`):__ Keep both near zero when you want weak return-level autocorrelation. Increase carefully only when you explicitly want stronger carry-over.
* __Mixture smoothness (`k`, `gamma_sharpen`):__ Larger `k` smooths more and can narrow distributions. Larger `gamma_sharpen` concentrates mass and increases regime contrast.
* __Selection noise (`sigma_select`):__ Small positive values prevent deterministic collapse to one memory and improve diversity. Excessive values make the sampler unstable and degrade realistic structure.
* __Post-calibration (`match_mean`, `match_std`, `alpha_std`):__ Use these to align marginal moments after generation. Treat them as calibration controls, not as substitutes for dynamic mechanisms.

A practical workflow is: fit dynamics first, then use moment corrections lightly.


## Summary

This method builds synthetic market returns by combining modern Hopfield memory probabilities with a stateful sampling loop and a persistent latent volatility state.

The key implication is that stylized facts are controlled by separate mechanism blocks, so calibration can be performed in a structured sequence instead of trial-and-error parameter edits.

> __Key Takeaways:__
>
> * __Stateful sampling is necessary for clustering:__ A memoryless daily sampler can reproduce heavy tails and weak linear autocorrelation but usually misses persistent volatility regimes. Introducing a persistent latent volatility state adds a direct mechanism for clustered magnitudes without forcing return-level persistence.
> * __Autocorrelation targets should be split by metric:__ You should evaluate both return autocorrelation and magnitude autocorrelation when assessing realism. Matching only linear return autocorrelation hides failures in volatility dynamics.
> * __Moment matching is a calibration layer:__ Mean and standard-deviation corrections can improve cross-sectional calibration after synthesis. These corrections do not create clustering and can distort shape if overused.

___


### References
1. Hopfield JJ (1982), *Neural networks and physical systems with emergent collective computational abilities*, PNAS 79(8):2554-2558. DOI: https://doi.org/10.1073/pnas.79.8.2554
2. Ramsauer H, Schafl B, Lehner J, et al. (2021), *Hopfield Networks is All You Need*, ICLR 2021. arXiv: https://arxiv.org/abs/2008.02217
3. Cont R (2001), *Empirical properties of asset returns: stylized facts and statistical issues*, Quantitative Finance 1(2):223-236. DOI: https://doi.org/10.1080/713665670

Note: the generator equations in this notebook are method-specific to the L6d workflow and are documented to make the calibration logic reproducible.
